# NMF on ModulAir 00685

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00685
data = pd.read_csv('MOD-00685.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:17Z,577610996,2025-12-31T18:59:17Z,MOD-00685,51.2,-0.3,8.543,0.676,0.158,0.029,0.024,...,31.660,22.673,14328,14329,14330,14472.0,14497.0,14547.0,14522.0,12.94
2025-12-31T23:58:17Z,577610995,2025-12-31T18:58:17Z,MOD-00685,51.7,-0.3,9.063,0.708,0.125,0.023,0.030,...,32.586,21.543,14328,14329,14330,14472.0,14497.0,14547.0,14522.0,11.85
2025-12-31T23:57:17Z,577610993,2025-12-31T18:57:17Z,MOD-00685,51.6,-0.3,8.758,0.714,0.158,0.022,0.024,...,32.354,21.912,14328,14329,14330,14472.0,14497.0,14547.0,14522.0,13.39
2025-12-31T23:56:17Z,577610994,2025-12-31T18:56:17Z,MOD-00685,51.2,-0.3,9.603,0.839,0.190,0.033,0.023,...,32.133,22.318,14328,14329,14330,14472.0,14497.0,14547.0,14522.0,10.24
2025-12-31T23:55:17Z,577608926,2025-12-31T18:55:17Z,MOD-00685,51.1,-0.4,8.368,0.627,0.147,0.044,0.015,...,32.130,22.325,14328,14329,14330,14472.0,14497.0,14547.0,14522.0,15.26


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:17Z,2025-12-31T18:59:17Z,704.365,2.293,31.660,22.673,8.543,0.676,0.158,0.029,0.024,0.003
2025-12-31T23:58:17Z,2025-12-31T18:58:17Z,711.399,2.083,32.586,21.543,9.063,0.708,0.125,0.023,0.030,0.006
2025-12-31T23:57:17Z,2025-12-31T18:57:17Z,711.470,2.083,32.354,21.912,8.758,0.714,0.158,0.022,0.024,0.006
2025-12-31T23:56:17Z,2025-12-31T18:56:17Z,709.702,2.293,32.133,22.318,9.603,0.839,0.190,0.033,0.023,0.006
2025-12-31T23:55:17Z,2025-12-31T18:55:17Z,721.159,2.228,32.130,22.325,8.368,0.627,0.147,0.044,0.015,0.012


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:17Z,704.365,2.293,31.660,22.673,8.543,0.676,0.158,0.029,0.024,0.003
1,2025-12-31T18:58:17Z,711.399,2.083,32.586,21.543,9.063,0.708,0.125,0.023,0.030,0.006
2,2025-12-31T18:57:17Z,711.470,2.083,32.354,21.912,8.758,0.714,0.158,0.022,0.024,0.006
3,2025-12-31T18:56:17Z,709.702,2.293,32.133,22.318,9.603,0.839,0.190,0.033,0.023,0.006
4,2025-12-31T18:55:17Z,721.159,2.228,32.130,22.325,8.368,0.627,0.147,0.044,0.015,0.012


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:17,704.365,2.293,31.660,22.673,8.543,0.676,0.158,0.029,0.024,0.003
1,2025-12-31 18:58:17,711.399,2.083,32.586,21.543,9.063,0.708,0.125,0.023,0.030,0.006
2,2025-12-31 18:57:17,711.470,2.083,32.354,21.912,8.758,0.714,0.158,0.022,0.024,0.006
3,2025-12-31 18:56:17,709.702,2.293,32.133,22.318,9.603,0.839,0.190,0.033,0.023,0.006
4,2025-12-31 18:55:17,721.159,2.228,32.130,22.325,8.368,0.627,0.147,0.044,0.015,0.012


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-03-31 20:00:00,892.48,32.71,21.42,1.92,27.09,2.76,1.24,0.49,0.72,0.57
1,2025-03-31 21:00:00,909.38,36.60,20.52,2.37,33.38,4.49,1.52,0.49,0.73,0.63
2,2025-03-31 22:00:00,908.36,27.52,28.84,2.45,26.71,5.29,1.53,0.42,0.52,0.41
3,2025-03-31 23:00:00,850.11,20.29,39.52,2.48,17.73,3.84,1.06,0.25,0.29,0.18
4,2025-04-01 00:00:00,760.03,23.20,42.68,2.34,18.32,3.75,1.01,0.24,0.26,0.17
...,...,...,...,...,...,...,...,...,...,...,...
6547,2025-12-31 14:00:00,677.24,30.93,28.43,2.09,10.06,0.58,0.14,0.03,0.02,0.01
6548,2025-12-31 15:00:00,672.58,30.98,28.47,2.44,8.97,0.56,0.14,0.02,0.02,0.01
6549,2025-12-31 16:00:00,699.01,31.85,25.50,2.18,10.62,0.71,0.18,0.03,0.03,0.02
6550,2025-12-31 17:00:00,717.85,32.84,21.98,2.02,11.45,0.84,0.21,0.04,0.03,0.02


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,892.48,32.71,21.42,1.92,27.09,2.76,1.24,0.49,0.72,0.57
2025-03-31 21:00:00,909.38,36.60,20.52,2.37,33.38,4.49,1.52,0.49,0.73,0.63
2025-03-31 22:00:00,908.36,27.52,28.84,2.45,26.71,5.29,1.53,0.42,0.52,0.41
2025-03-31 23:00:00,850.11,20.29,39.52,2.48,17.73,3.84,1.06,0.25,0.29,0.18
2025-04-01 00:00:00,760.03,23.20,42.68,2.34,18.32,3.75,1.01,0.24,0.26,0.17


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.585559,0.664973,0.251852,0.054779,0.310452,0.068385,0.067172,0.087189,0.134579,0.148825
2025-03-31 21:00:00,0.596647,0.744054,0.241270,0.067618,0.382535,0.111249,0.082340,0.087189,0.136449,0.164491
2025-03-31 22:00:00,0.595978,0.559463,0.339095,0.069900,0.306097,0.131070,0.082882,0.074733,0.097196,0.107050
2025-03-31 23:00:00,0.557760,0.412482,0.464668,0.070756,0.203186,0.095144,0.057421,0.044484,0.054206,0.046997
2025-04-01 00:00:00,0.498658,0.471641,0.501822,0.066762,0.209947,0.092914,0.054713,0.042705,0.048598,0.044386


In [11]:
data.to_csv(r'MOD-00685_timeseries_hourly_scaled.csv')

In [12]:
start = data.index.min()

end = data.index.min()

print(start, end)

2025-03-31 20:00:00 2025-03-31 20:00:00


## Implementing NMF

In [13]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [14]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.565112,0.673652,0.264607,0.068448,0.299014,0.148883,0.101434,0.078087,0.078781,0.068886
2025-03-31 21:00:00,0.609418,0.739859,0.237600,0.072285,0.358266,0.175268,0.117650,0.089127,0.089228,0.077511
2025-03-31 22:00:00,0.573743,0.568709,0.351503,0.071393,0.311188,0.151466,0.102511,0.078944,0.080031,0.070153
2025-03-31 23:00:00,0.529294,0.423910,0.479254,0.069559,0.217841,0.093553,0.061401,0.048527,0.051380,0.046951
2025-04-01 00:00:00,0.532831,0.457874,0.484045,0.070309,0.194282,0.084362,0.056304,0.045450,0.048604,0.044820
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.464772,0.620669,0.323776,0.058136,0.103741,0.019843,0.006323,0.005037,0.008444,0.011184
2025-12-31 15:00:00,0.460756,0.622509,0.325524,0.057922,0.092683,0.017222,0.005578,0.005003,0.008491,0.011247
2025-12-31 16:00:00,0.466698,0.644482,0.295959,0.057436,0.116643,0.023153,0.007272,0.004964,0.008125,0.010804


In [15]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.059736,0.059437,0.036184,0.058349
1,0.065002,0.072100,0.024748,0.067008
2,0.049283,0.061263,0.068556,0.058733
3,0.037451,0.040504,0.113738,0.034173
4,0.041245,0.035451,0.112545,0.031664
...,...,...,...,...
6477,0.059639,0.017775,0.053443,0.000000
6478,0.059992,0.015427,0.053710,0.000000
6479,0.061759,0.020740,0.043916,0.000000
6480,0.063944,0.024339,0.032166,0.000000


In [16]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [17]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,5.338800,10.183775,2.344966,0.651061,0.175056,0.000000,0.009501,0.052512,0.093062,0.126577
Feature 2,2.722848,0.749447,0.000000,0.221947,4.754765,1.116362,0.318768,0.025303,0.000000,0.000000
Feature 3,1.833254,0.000000,3.441523,0.287454,0.164394,0.000000,0.001688,0.027235,0.054143,0.068018
Feature 4,0.308859,0.355970,0.000000,0.102201,0.000000,1.414437,1.402923,1.241863,1.221330,1.008835


In [18]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-03-31 20:00:00,0.059736,0.059437,0.036184,0.058349
2025-03-31 21:00:00,0.065002,0.072100,0.024748,0.067008
2025-03-31 22:00:00,0.049283,0.061263,0.068556,0.058733
2025-03-31 23:00:00,0.037451,0.040504,0.113738,0.034173
2025-04-01 00:00:00,0.041245,0.035451,0.112545,0.031664
...,...,...,...,...
2025-12-31 14:00:00,0.059639,0.017775,0.053443,0.000000
2025-12-31 15:00:00,0.059992,0.015427,0.053710,0.000000
2025-12-31 16:00:00,0.061759,0.020740,0.043916,0.000000


In [19]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,5.338800,10.183775,2.344966,0.651061,0.175056,0.000000,0.009501,0.052512,0.093062,0.126577
Factor 2,2.722848,0.749447,0.000000,0.221947,4.754765,1.116362,0.318768,0.025303,0.000000,0.000000
Factor 3,1.833254,0.000000,3.441523,0.287454,0.164394,0.000000,0.001688,0.027235,0.054143,0.068018
Factor 4,0.308859,0.355970,0.000000,0.102201,0.000000,1.414437,1.402923,1.241863,1.221330,1.008835


In [20]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.510515,0.160197,0.308716,0.007324,0.013247
no,0.486813,0.102107,0.378513,0.018951,0.013615
no2,0.953860,0.043190,0.000000,0.008269,-0.005319
o3,0.281437,0.000000,0.727390,0.000000,-0.008827
bin0,0.052640,0.879703,0.087056,0.000000,-0.019399
bin1,0.000000,0.742721,0.000000,0.379302,-0.122023
bin2,0.017354,0.358233,0.005429,0.635486,-0.016502
bin3,0.124297,0.036851,0.113530,0.728998,-0.003676
bin4,0.188866,0.000000,0.193506,0.614699,0.002929
bin5,0.249269,0.000000,0.235891,0.492701,0.022138


In [21]:
len(data)

6482

In [ ]:
results.to_csv(r'MOD-00685_4_factor_results.csv')
comp.to_csv(r'MOD-00685_4_factor_comp.csv')
res.to_csv(r'MOD-00685_4_factor_resid.csv')